In [0]:
%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, row_number
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@adlsg2rag.dfs.core.windows.net/sqlserver/employees/load_date=2026-03-13/"

In [0]:
df = spark.read.format("parquet").load(bronze_path)

df.printSchema()

In [0]:
# PK validation
df_clean = df.filter(col("EmployeeID").isNotNull() & (col("EmployeeID") != 0))

In [0]:
# Date validation
df_clean = df_clean.withColumn("HireDate", to_date(col("HireDate")))
df_clean = df_clean.filter(col("HireDate").isNotNull())

In [0]:
# Processing timestamp
df_clean = df_clean.withColumn("processed_ts", current_timestamp())

In [0]:
# Duplicate check
w = Window.partitionBy("EmployeeID").orderBy(col("processed_ts").desc())
df_clean = (
    df_clean.withColumn("rn", row_number().over(w))
            .filter(col("rn") == 1)
            .drop("rn")
)

In [0]:
# Select and rename columns
df_clean = df_clean.select(
    col("EmployeeID").alias("src_EmployeeID"),
    col("EmployeeName").alias("src_EmployeeName"),
    col("Department").alias("src_Department"),
    col("HireDate").alias("src_HireDate"),
    col("Salary").alias("src_Salary"),
    col("processed_ts")
)


In [0]:
catalog_name = 'adbrag'
schema_name = 'silver'

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.employees_silver
(
    src_EmployeeID INT,
    src_EmployeeName STRING,
    src_Department STRING,
    src_HireDate DATE,
    src_Salary DECIMAL(10,2),
    processed_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_EmployeeID)
""")



In [0]:
df_clean.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{schema_name}.employees_silver"
)